# CondAptNet — Stage 1 Training (Colab Premium / A100)

**Before running:** Runtime → Change runtime type → A100 GPU  
**Flow:** GPU check → clone repo → install deps → vienna cache → confirm splits → train → evaluate → push checkpoint

---

## Cell 1 — GPU check

In [ ]:
import torch

assert torch.cuda.is_available(), (
    "No GPU. Go to Runtime → Change runtime type → GPU (A100)."
)

gpu = torch.cuda.get_device_properties(0)
vram_gb = gpu.total_memory / 1024 ** 3
print(f"GPU  : {gpu.name}")
print(f"VRAM : {vram_gb:.1f} GB")
print(f"CUDA : {torch.version.cuda}")

if vram_gb >= 35:
    BATCH_SIZE   = 32
    MAX_PROT_LEN = 1024
    USE_AMP      = True   # BF16 halves activation memory — required at these settings on A100
    print("\nA100 detected → batch_size=32, max_prot_len=1024, AMP=BF16")
elif vram_gb >= 14:
    BATCH_SIZE   = 16
    MAX_PROT_LEN = 512
    USE_AMP      = False
    print("\nV100/T4 detected → batch_size=16, max_prot_len=512")
else:
    BATCH_SIZE   = 8
    MAX_PROT_LEN = 256
    USE_AMP      = False
    print("\nSmall GPU → batch_size=8, max_prot_len=256")

## Cell 2 — Clone repo

In [ ]:
import os, subprocess

REPO        = "shivanshb828/CondAptNet"
PROJECT_DIR = "/content/CondAptNet"

os.chdir("/content")  # always reset cwd first — avoids "no such file or directory" on restart

if os.path.isdir(PROJECT_DIR):
    print("Repo already present — skipping clone.")
else:
    print("Cloning repo...")
    subprocess.run([
        "git", "clone", f"https://github.com/{REPO}.git", PROJECT_DIR,
    ], check=True)
    print("Done.")

os.chdir(PROJECT_DIR)
!git log --oneline -3

## Cell 3 — Install dependencies

In [ ]:
import sys

!pip install fair-esm -q
!pip install ViennaRNA -q
# scikit-learn, scipy, pandas, numpy are pre-installed on Colab

# Verify ESM loads
import esm
print("fair-esm OK")

# Verify ViennaRNA (optional — zero fallback if missing)
try:
    import RNA
    print("ViennaRNA OK")
except ImportError:
    print("ViennaRNA not available — secondary structure features will be zero vectors (training still works)")

sys.path.insert(0, PROJECT_DIR)
import config
print(f"CondAptNet device: {config.DEVICE}")   # must print: cuda

## Cell 4 — Generate ViennaRNA cache  *(run once, ~10 min)*

In [ ]:
import os

VIENNA_CACHE = "data/processed/vienna_cache.pkl"

if os.path.exists(VIENNA_CACHE):
    import pickle
    with open(VIENNA_CACHE, "rb") as f:
        vc = pickle.load(f)
    print(f"Vienna cache already exists: {len(vc):,} entries — skipping.")
else:
    print("Generating ViennaRNA cache (run once)...")
    !python scripts/data/vienna_features.py \
        --input data/processed/master_dataset_cleaned.csv

## Cell 5 — Confirm augmented splits

In [ ]:
import os, pandas as pd

splits = {
    "tier1_train": "data/augmented/tier1_train.csv",
    "val"        : "data/augmented/val.csv",
    "test"       : "data/augmented/test.csv",
}

missing = [k for k, p in splits.items() if not os.path.exists(p)]
if missing:
    print(f"Missing splits: {missing} — regenerating...")
    !python scripts/data/augment.py

for name, path in splits.items():
    df   = pd.read_csv(path)
    ready = df["aptamer_sequence"].notna() & df["protein_sequence"].notna()
    print(f"{name:15s}: {len(df):6,} rows  ({ready.sum():,} training-ready)")

## Cell 6 — Resume check

In [ ]:
import glob, torch

CHECKPOINT_DIR = "models/checkpoints/pretrain"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

existing = sorted(glob.glob(f"{CHECKPOINT_DIR}/epoch_*.pt"))

if existing:
    latest = existing[-1]
    # weights_only=True: block pickle-deserialization RCE from a malicious
    # checkpoint. Our checkpoints hold only tensors/dicts/numbers/strings.
    meta   = torch.load(latest, map_location="cpu", weights_only=True)
    print(f"Checkpoint : {latest}")
    print(f"Last epoch : {meta.get('epoch', '?')}")
    print(f"Best MCC   : {meta.get('best_val_mcc', meta.get('val_mcc', float('nan'))):.4f}")
    print(f"Patience   : {meta.get('patience_count', '?')}")
    RESUME_FLAG = "--resume"
    print("\nWill resume from next epoch.")
else:
    RESUME_FLAG = ""
    print("No checkpoint found — starting fresh.")

## Cell 7 — Train

In [ ]:
import subprocess, os

amp_flag = ["--use-amp"] if USE_AMP else []

cmd = [
    "python", "scripts/training/train.py",
    "--max-epochs",      "50",
    "--batch-size",      str(BATCH_SIZE),
    "--max-prot-len",    str(MAX_PROT_LEN),
    "--prot-max-tokens", str(MAX_PROT_LEN),
    "--checkpoint-dir",  CHECKPOINT_DIR,
] + amp_flag
if RESUME_FLAG:
    cmd.append(RESUME_FLAG)

env = os.environ.copy()
env["PYTHONUNBUFFERED"]   = "1"
env["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

print("Command:", " ".join(cmd))
print("-" * 60)

proc = subprocess.Popen(
    cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, env=env,
)
for line in proc.stdout:
    print(line, end="")
proc.wait()

print("-" * 60)
print(f"Training finished (exit code {proc.returncode}).")

## Cell 8 — Evaluate

In [ ]:
import os

best_ckpt = os.path.join(CHECKPOINT_DIR, "best.pt")

if not os.path.exists(best_ckpt):
    print(f"No best checkpoint at {best_ckpt} — run Cell 7 first.")
else:
    for split in ("val", "test"):
        print(f"\n{'='*55}")
        print(f"  {split.upper()} split")
        print(f"{'='*55}")
        !python scripts/evaluation/evaluate.py \
            --checkpoint {best_ckpt} \
            --split {split} \
            --max-prot-len {MAX_PROT_LEN}

## Cell 9 — Push checkpoint back to GitHub

In [ ]:
import os, subprocess

# Only needed here for pushing — paste a classic PAT (ghp_...) from
# github.com/settings/tokens → classic → repo scope
GITHUB_TOKEN = "ghp_xxxxxxxxxxxxxxxxxxxx"   # ← paste your token

REPO = "shivanshb828/CondAptNet"

!git config user.email "coolshivansh7@gmail.com"
!git config user.name  "shivanshb828"
!git remote set-url origin https://{GITHUB_TOKEN}@github.com/{REPO}.git

files_to_add = [
    "models/checkpoints/pretrain/best.pt",
    "data/processed/vienna_cache.pkl",
]
existing_files = [f for f in files_to_add if os.path.exists(f)]

if not existing_files:
    print("Nothing to push — run Cells 7–8 first.")
else:
    for f in existing_files:
        size_mb = os.path.getsize(f) / 1e6
        if size_mb > 100:
            print(f"WARNING: {f} is {size_mb:.0f} MB — exceeds GitHub's 100MB limit.")
            print("Run Cell 9b (Git LFS) first.")

    !git add {" ".join(existing_files)}
    !git commit -m "chore: Stage 1 checkpoint + vienna cache from Colab A100"
    !git push
    print("Pushed.")

## Cell 9b — Git LFS setup *(only if checkpoint > 100 MB)*
Run this **before** Cell 9 if the checkpoint file is over 100 MB.

In [ ]:
!apt-get install -y git-lfs -q
!git lfs install
!git lfs track "*.pt"
!git add .gitattributes
!git commit -m "chore: track .pt files with Git LFS"
print("Git LFS configured. Now run Cell 9 to push the checkpoint.")